### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GPT_4_MINI_API_KEY"] = os.getenv("GPT_4_MINI_API_KEY")

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("GPT_4_MINI_API_KEY"),
    base_url="https://models.inference.ai.azure.com"
)

agent = create_agent(
    model=llm,
    # tools=[],
    checkpointer=InMemorySaver(),   # Storing in Hard disk
    middleware=[
        SummarizationMiddleware(
            model=llm,  # pick low power llm as it required only text summarization
            trigger=[("messages", 10)],    #Summarize for every 10 messages
            keep=("messages", 4)    # Even after summarization keep last 4 messages
        )
    ]
)

In [3]:
# Run with thread id to enable memory
config = {"configurable":{"thread_id":"test-1"}}

* Purpose of the config parameter: it passes runtime options into agent.invoke (like which conversation/session to use). Middleware and the checkpointer read values from config to store/retrieve conversation history, enable summarization, etc.

* What "thread" means here: "thread-id" is a conversation/session identifier (not an OS/thread). It isolates memory and middleware state per conversation so multiple sessions/users don't mix.

* If you omit or reuse the same thread-id, memory/summaries will be unavailable or shared across those calls. Use distinct thread-ids for separate conversations.

```python
# simple example
config = {"configurable": {"thread-id": "test-1"}}
response = agent.invoke({"messages":[HumanMessage(content="Hi")]}, config)
# use "test-2" for a separate conversation
```

In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='55278297-f575-47a4-8117-690f7f6a2693'), AIMessage(content='2 + 2 = 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 14, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DLq4GkoVE98jZyKHSeDFNqdWhvoXD', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1072-15f0-7b70-9ef0-34fab85aaeb9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 9, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

### Token Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


llm = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("GPT_4_MINI_API_KEY"),
    base_url="https://models.inference.ai.azure.com"
)

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars ≈ token

In [6]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~122 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='53fc4908-7549-4d31-918b-467685b66d93'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 53, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DLq4P3oXREtOPXLzOle0jPwggyvaJ', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1072-44ba-7131-b168-d0cb629bc8b7-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_g74QuZRu3zIrqoXGimvnriUc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'to

### Fraction

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 32000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~70 tokens (0.2188%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a4b7f3f2-0eaa-48f3-b354-18b21569bfee'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 44, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DLq5XnsvPcdtncwtqyH3WnJKJFvu1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1073-5589-7352-9c87-831d1283f374-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_7YbSsEL4ydNi9NvGAs8ZxXJ9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 16, 'to

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [9]:
agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    # Valid values: "approve", "edit", "reject"
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [10]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to john.doe@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'")]},
    config=config
)

In [11]:
result

{'messages': [HumanMessage(content="Send an email to john.doe@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='270f1553-f546-4903-a37a-1dca3d6bd568'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 104, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DLq6O4jTj6QW5SyX2poggAp5YiFcK', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1074-24ce-71e2-90f8-c271fbd56f65-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john.doe@example.com', 'subject': 'Meeting', 'body': "Let's meet tomorrow at 10 

In [12]:
from langgraph.types import Command

# Approve
if "__interrupt__" in result:
    print("Paused. Approving command...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}   # valid: "approve", "edit", "reject"
                ]
            }
        ),
        config=config
    )

    if result.get('messages') and len(result['messages']) > 0:
        print(f"Result: {result['messages'][-1].content}")
    else:
        print(f"Result: {result}")
else:
    print("No interrupt detected.")
    print(f"Result: {result}")

Paused. Approving command...
Result: The email with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.' has been sent to john.doe@example.com.


In [13]:
result

{'messages': [HumanMessage(content="Send an email to john.doe@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='270f1553-f546-4903-a37a-1dca3d6bd568'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 104, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DLq6O4jTj6QW5SyX2poggAp5YiFcK', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1074-24ce-71e2-90f8-c271fbd56f65-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john.doe@example.com', 'subject': 'Meeting', 'body': "Let's meet tomorrow at 10 

### Reject

In [14]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to john.doe@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'")]},
    config=config
)

In [15]:
from langgraph.types import Command

# Approve
if "__interrupt__" in result:
    print("Paused. Approving command...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "reject"}   # valid: "approve", "edit", "reject"
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused. Approving command...
Result: It seems that I am currently unable to send the email. Is there anything else I can assist you with?


In [16]:
config = {"configurable": {"thread_id": "test-edit"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to john.doe@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'")]},
    config=config
)

In [17]:
from langgraph.types import Command

# Approve
if "__interrupt__" in result:
    print("Paused. Approving command...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused. Approving command...
Result: The email has been sent to john.doe@example.com with the subject 'Meeting' and the body 'Let's meet tomorrow at 10 AM.'
